In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv('compas-scores-two-years_v1.csv')

#Artificial Preprocessing (HAND IN ONLY THE PREPROCESSED DATA)

In [ ]:
df.columns

Index(['id', 'name', 'first', 'last', 'compas_screening_date', 'sex', 'dob',
       'age', 'age_cat', 'race', 'juv_fel_count', 'decile_score',
       'juv_misd_count', 'juv_other_count', 'priors_count',
       'days_b_screening_arrest', 'c_jail_in', 'c_jail_out', 'c_case_number',
       'c_offense_date', 'c_arrest_date', 'c_days_from_compas',
       'c_charge_degree', 'c_charge_desc', 'is_recid', 'r_case_number',
       'r_charge_degree', 'r_days_from_arrest', 'r_offense_date',
       'r_charge_desc', 'r_jail_in', 'r_jail_out', 'violent_recid',
       'is_violent_recid', 'vr_case_number', 'vr_charge_degree',
       'vr_offense_date', 'vr_charge_desc', 'type_of_assessment',
       'decile_score.1', 'score_text', 'screening_date',
       'v_type_of_assessment', 'v_decile_score', 'v_score_text',
       'v_screening_date', 'in_custody', 'out_custody', 'priors_count.1',
       'start', 'end', 'event', 'two_year_recid'],
      dtype='object')

In [ ]:
columns_to_keep = [
    'race', 'decile_score', 'score_text', 'two_year_recid',
    'priors_count', 'age', 'sex', 'c_charge_degree', 'days_b_screening_arrest'
]
df = df[columns_to_keep]

df = df[
    (df['days_b_screening_arrest'] >= -30) &
    (df['days_b_screening_arrest'] <= 30) &
    (df['c_charge_degree'] != 'O') &
    (df['score_text'].isna() == False) &
    (df['race'].isin(['Caucasian', 'African-American']))
]



In [ ]:
df.head()

,race,decile_score,score_text,two_year_recid,priors_count,age,sex,c_charge_degree,days_b_screening_arrest,predicted_recid
1,African-American,3.0,Low,1.0,0.0,34.0,Male,F,-1.0,0
2,African-American,4.0,Low,1.0,4.0,24.0,Male,F,-1.0,0
6,Caucasian,6.0,Medium,1.0,14.0,41.0,Male,F,-1.0,1
8,Caucasian,1.0,Low,0.0,0.0,39.0,Female,M,-1.0,0
10,Caucasian,4.0,Low,0.0,0.0,27.0,Male,F,-1.0,0


#Examine Discrimination

In [ ]:
df['predicted_recid'] = (df['decile_score'] >= 5).astype(int)

fairness_summary = []

for race in df['race'].unique():
    race_df = df[df['race'] == race]

    total_count = len(race_df)
    actual_pos = race_df[race_df['two_year_recid'] == 1]  # People who actually reoffended
    actual_neg = race_df[race_df['two_year_recid'] == 0]  # People who did NOT reoffend
    pred_pos = race_df[race_df['predicted_recid'] == 1]    # People flagged as High/Med risk

    # Protect against division-by-zero errors for tiny sub-samples
    if total_count < 10 or len(actual_pos) == 0 or len(actual_neg) == 0 or len(pred_pos) == 0:
        continue

    # Independence (Demographic Parity / Selection Rate)
    selection_rate = (race_df['predicted_recid'] == 1).mean() * 100

    # Separation (Equalized Odds: Split into FPR and TPR)
    fpr = (actual_neg['predicted_recid'] == 1).mean() * 100
    tpr = (actual_pos['predicted_recid'] == 1).mean() * 100

    # Sufficiency (Predictive Parity / Precision / Calibration)
    precision = (pred_pos['two_year_recid'] == 1).mean() * 100

    fairness_summary.append({
        'Race': race,
        'Sample Size': total_count,
        'Selection Rate (%)': round(selection_rate, 1),
        'False Positive Rate (%)': round(fpr, 1),
        'True Positive Rate (%)': round(tpr, 1),
        'Precision (%)': round(precision, 1)
    })

# Display results nicely in a table ordered by sample size
results_df = pd.DataFrame(fairness_summary).sort_values(by='Sample Size', ascending=False)
print("\n--- COMPAS GROUP FAIRNESS METRICS ---")
print(results_df.to_string(index=False))


--- COMPAS GROUP FAIRNESS METRICS ---
            Race  Sample Size  Selection Rate (%)  False Positive Rate (%)  True Positive Rate (%)  Precision (%)
African-American         3175                57.6                     42.3                    71.5           65.0
       Caucasian         2103                33.1                     22.0                    50.4           59.5


In [ ]:
# Isolate selection rates for African-American and Caucasian groups
sr_black = results_df.loc[results_df['Race'] == 'African-American', 'Selection Rate (%)'].values[0] / 100
sr_white = results_df.loc[results_df['Race'] == 'Caucasian', 'Selection Rate (%)'].values[0] / 100

disparate_impact = sr_black / sr_white

print(f"Disparate Impact Ratio (Black vs White): {disparate_impact:.3f}")
if disparate_impact > 1.25 or disparate_impact < 0.8:
    print("Legally significant adverse impact detected.")
else:
    print("Within acceptable statistical parity thresholds.")

Disparate Impact Ratio (Black vs White): 1.740
Legally significant adverse impact detected.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Check the average priors count across races
proxy_check = df.groupby('race')[['priors_count', 'decile_score']].mean().reset_index()
print("\n--- PROXY VARIABLE ANALYSIS ---")
print(proxy_check.to_string(index=False))

# Check correlation between priors and decile score
correlation = df['priors_count'].corr(df['decile_score'])
print(f"\nOverall Correlation between Priors Count and COMPAS Decile Score: {correlation:.3f}")


--- PROXY VARIABLE ANALYSIS ---
            race  priors_count  decile_score
African-American      4.238110      5.276850
       Caucasian      2.289111      3.635283

Overall Correlation between Priors Count and COMPAS Decile Score: 0.440


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

for race in ['African-American', 'Caucasian']:
    race_df = df[df['race'] == race]
    auc = roc_auc_score(race_df['two_year_recid'], race_df['decile_score'])
    print(f"{race} Model AUC (Predictive Power): {auc:.3f}")

African-American Model AUC (Predictive Power): 0.704
Caucasian Model AUC (Predictive Power): 0.693


#Graphs

#Conclusions